# Lifting accuracy — ViraLift tblastn engine

One notebook for every virus; the dataset is a parameter. It **does not run tblastn**: it reads a
run folder produced by

```
python -m validation.valkit.runner lifting_accuracy
```

so the numbers here are exactly the numbers in that run's `summary.json`, and a run made inside
the Docker image can be read on a laptop. Scoring rules: `validation/METHODOLOGY.md`
(truth = the record's own annotation after alias normalisation; denominator `R ∩ truth`;
correct = IoU ≥ 0.90 or both boundaries within ±6 bp; codon check only for genes whose
reference CDS is itself codon-clean).

Cells marked **[auto]** are generated from the data by the notebook itself.

In [ ]:
# Parameters (overridden by validation.valkit.notebooks --param NAME=VALUE)
DATASET = "prrs"          # label in validation/configs/datasets.json
RUN_ID = "latest"         # folder under validation/outputs/, or "latest" for the newest lifting_accuracy run


In [ ]:
from pathlib import Path
import json, sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
for c in [ROOT, *ROOT.parents]:
    if (c / "app" / "src").exists():
        ROOT = c
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from validation.valkit.paths import OUTPUTS
from validation.valkit.report import adjudicate_failures

if RUN_ID == "latest":
    runs = sorted(p for p in OUTPUTS.glob("*_lifting_accuracy") if (p / "summary.json").exists())
    if not runs:
        raise SystemExit("no lifting_accuracy run under validation/outputs/ — run the runner first")
    RUN_DIR = runs[-1]
else:
    RUN_DIR = OUTPUTS / RUN_ID
DS = RUN_DIR / DATASET
manifest = json.loads((RUN_DIR / "manifest.json").read_text())
summary = json.loads((RUN_DIR / "summary.json").read_text())["datasets"][DATASET]
print("run     :", RUN_DIR.name)
print("engine  :", manifest["engine"], "| commit", manifest["environment"]["commit"],
      "| tblastn", manifest["environment"]["tblastn"], "| python", manifest["environment"]["python"],
      "| container" if manifest["environment"]["in_container"] else "| host")
print("dataset :", summary["virus"], f"({summary['feature_level']})", "ref", summary["reference"], "query", summary["query"])

per_pred = pd.read_csv(DS / "per_prediction.tsv", sep="\t")
per_gene = pd.read_csv(DS / "per_gene.tsv", sep="\t")
overall = pd.read_csv(DS / "overall.tsv", sep="\t")
overall

## Per-gene accuracy

In [ ]:
cols = ["gene", "total", "exact", "coord_only", "failed", "no_hit",
        "extra_predictions_without_truth", "exact_pct", "coord_only_pct", "failed_pct", "accuracy_pct"]
per_gene[cols]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
x = range(len(per_gene))
ax.bar(x, per_gene["exact"], label="exact", color="#2a9d8f")
ax.bar(x, per_gene["coord_only"], bottom=per_gene["exact"], label="coord-only (IoU≥0.90 or ±6 bp)", color="#e9c46a")
ax.bar(x, per_gene["failed"], bottom=per_gene["exact"] + per_gene["coord_only"], label="failed", color="#e76f51")
ax.set_xticks(list(x)); ax.set_xticklabels(per_gene["gene"], rotation=45, ha="right")
ax.set_ylabel("records with the gene in truth"); ax.set_title(f"{summary['virus']} — lifting accuracy per gene ({RUN_DIR.name})")
ax.legend(fontsize=8, loc="lower right"); fig.tight_layout()
(DS / "per_gene_accuracy.png").write_bytes(b"")  # placeholder so the path exists before savefig
fig.savefig(DS / "per_gene_accuracy.png", dpi=160, bbox_inches="tight")
plt.show()

## Cases that are not exact

In [ ]:
eval_rows = per_pred.merge(
    pd.read_csv(DS / "truth_presence.tsv", sep="\t").rename(columns={"gene": "ref_name"}),
    on=["record_id", "ref_name"], how="left")
eval_rows = eval_rows[eval_rows["has_truth_gene"].fillna(False)]
diag_cols = ["record_id", "ref_name", "status", "pred_start", "pred_end", "truth_start", "truth_end",
             "delta_start", "delta_end", "pred_minus_ref_len", "truth_minus_ref_len", "iou",
             "has_start_codon", "has_stop_codon", "in_frame", "rescue_action", "failure_mode"]
coord_only_cases = eval_rows[eval_rows["coord_correct"] & ~eval_rows["exact_match"]]
failed_cases = eval_rows[~eval_rows["coord_correct"]]
print(f"coord-only: {len(coord_only_cases)}   failed: {len(failed_cases)}")
coord_only_cases[diag_cols].to_csv(DS / "coord_only_cases.tsv", sep="\t", index=False)
failed_cases[diag_cols].to_csv(DS / "failed_cases.tsv", sep="\t", index=False)

### Coord-only — where do the boundaries sit relative to truth and reference?

In [ ]:
if len(coord_only_cases):
    display(coord_only_cases.groupby("ref_name").agg(
        n=("record_id", "size"),
        delta_start_values=("delta_start", lambda s: sorted(s.value_counts().to_dict().items(), key=lambda kv: -kv[1])[:4]),
        delta_end_values=("delta_end", lambda s: sorted(s.value_counts().to_dict().items(), key=lambda kv: -kv[1])[:4]),
        pred_follows_ref_len=("pred_minus_ref_len", lambda s: int((s == 0).sum())),
        truth_follows_ref_len=("truth_minus_ref_len", lambda s: int((s == 0).sum())),
    ))

### Failed — every case, with a generic first-pass adjudication

In [ ]:
if len(failed_cases):
    adj = adjudicate_failures(failed_cases)
    adj.to_csv(DS / "failure_adjudication.tsv", sep="\t", index=False)
    display(adj[diag_cols + ["blame", "cause"]])
    display(adj.groupby(["ref_name", "blame", "cause"]).size().rename("n").reset_index())

## Extra predictions without same-name truth (name gap / granularity)

In [ ]:
extra = per_pred.merge(
    pd.read_csv(DS / "truth_presence.tsv", sep="\t").rename(columns={"gene": "ref_name"}),
    on=["record_id", "ref_name"], how="left")
extra = extra[~extra["has_truth_gene"].fillna(False)]
extra[["record_id", "ref_name", "status", "pred_start", "pred_end", "best_overlap_name", "best_iou", "failure_mode"]].to_csv(
    DS / "extra_without_truth_cases.tsv", sep="\t", index=False)
extra.groupby(["ref_name", "best_overlap_name"], dropna=False).size().rename("n").reset_index()

## Summary [auto]

In [ ]:
from IPython.display import Markdown
row = overall[overall["virus"] == DATASET].iloc[0]
worst = per_gene.sort_values("accuracy_pct").iloc[0]
lines = [
    f"**{summary['virus']}**, {summary['records_scored']} records scored against reference `{summary['reference']}`, "
    f"engine `{manifest['engine']}` at commit `{manifest['environment']['commit']}` ({manifest['environment']['tblastn']}).",
    "",
    f"- Denominator: **{int(row['total'])}** (record, gene) pairs where the reference gene is present in the record's own annotation.",
    f"- Exact: **{int(row['exact'])}** ({row['exact_pct']}%); coord-only: **{int(row['coord_only'])}** ({row['coord_only_pct']}%); "
    f"failed: **{int(row['failed'])}** ({row['failed_pct']}%), of which no_hit: {int(row['no_hit'])}.",
    f"- Accuracy (exact + coord-only): **{row['accuracy_pct']}%**.",
    f"- Lowest gene: **{worst['gene']}** at {worst['accuracy_pct']}% ({int(worst['failed'])} failed of {int(worst['total'])}).",
    f"- Predictions with no same-name truth (not in the denominator): {int(per_gene['extra_predictions_without_truth'].sum())}.",
]
if len(failed_cases):
    blame = adjudicate_failures(failed_cases)["blame"].value_counts().to_dict()
    lines.append(f"- First-pass blame of the failures: {blame} (see failure_adjudication.tsv; `review` means a human must decide).")
Markdown("\n".join(lines))